# PyTorch Autograd 기본

PyTorch 의 자동 미분(autograd) 동작을 작은 수식으로 확인하는 노트북입니다. 변수에 `requires_grad=True` 를 설정한 뒤 손실을 만들고 `.backward()` 를 호출하면 각 변수의 `.grad` 에 그래디언트가 저장됩니다.

## 학습 목표
- `requires_grad=True` 텐서로 계산 그래프가 자동 구성된다는 점 이해
- `loss.backward()` 호출 시 역전파가 일어나는 흐름
- `.grad` 속성에서 스칼라 손실에 대한 편미분 값 확인
- `grad_fn` 으로 텐서가 어떤 연산에서 만들어졌는지 추적

## 사용 라이브러리
- `torch`

## 다루는 수식
- `pred = 3a³ − b²`
- `cost = (pred − label)²`

In [1]:
import torch

a = torch.tensor([2.], requires_grad=True)
b = torch.tensor([6.], requires_grad=True)

print("[1] 입력 텐서 생성")
print("  a =", a, "| requires_grad =", a.requires_grad)
print("  b =", b, "| requires_grad =", b.requires_grad)
print("  -> 두 텐서는 미분 대상(leaf)이므로 이후 연산이 계산 그래프로 기록됩니다.")

[1] 입력 텐서 생성
  a = tensor([2.], requires_grad=True) | requires_grad = True
  b = tensor([6.], requires_grad=True) | requires_grad = True
  -> 두 텐서는 미분 대상(leaf)이므로 이후 연산이 계산 그래프로 기록됩니다.


## 입력 텐서와 계산 그래프 만들기

`requires_grad=True` 로 만든 텐서는 PyTorch 가 **계산 그래프** 를 자동으로 추적합니다. 이후 그 텐서들을 사용한 모든 연산 결과(`pred`, `cost` 등) 에는 `grad_fn` 이 붙어 역전파 시 사용됩니다.

| 표현 | 의미 |
|---|---|
| `requires_grad=True` | 미분 대상 변수임을 표시 |
| `pred.grad_fn` | 이 텐서를 만든 연산 (`<MulBackward0>`, `<PowBackward0>` 등) |
| `t.detach()` | 그래프에서 떼어내 그래디언트 추적 중단 |
| `with torch.no_grad():` | 블록 내부에서 그래프 추적 비활성화 (추론 시) |

## 순전파(forward) — 손실까지 계산하기

아래 세 줄이 차례대로 **계산 그래프를 쌓아 가는 과정**입니다. `a=2, b=6` 을 대입해 값을 직접 따라가 보면 흐름이 보입니다.

1. `pred = 3*a**3 - b**2` → `3·2³ − 6² = 24 − 36 = −12`
   결과 텐서에는 `grad_fn=<SubBackward0>` 가 붙습니다(마지막 연산이 뺄셈이라서). 이렇게 "어떤 연산으로 만들어졌는지"가 노드로 기록됩니다.
2. `label = torch.tensor([2.])` → 정답값. `requires_grad` 가 없으므로 **미분 대상이 아닌 상수**입니다.
3. `cost = (pred - label)**2` → `(−12 − 2)² = (−14)² = 196`
   역전파의 출발점이 되는 **스칼라 손실**이며 `grad_fn=<PowBackward0>` 이 붙습니다.

각 줄을 실행할 때마다 PyTorch 가 연산을 그래프에 노드로 기록해 두고, 나중에 `backward()` 가 이 그래프를 **거꾸로** 따라가며 그래디언트를 채웁니다.

In [2]:
pred = 3*a**3 - b**2

print("[2] 순전파 - 예측값 pred = 3a^3 - b^2")
print("  3*2^3 - 6^2 = 24 - 36 =", pred.item())
print("  pred.grad_fn =", pred.grad_fn, "(마지막 연산이 뺄셈이라 SubBackward)")

[2] 순전파 - 예측값 pred = 3a^3 - b^2
  3*2^3 - 6^2 = 24 - 36 = -12.0
  pred.grad_fn = <SubBackward0 object at 0x74cb88592620> (마지막 연산이 뺄셈이라 SubBackward)


In [3]:
label = torch.tensor([2.])

print("[3] 정답값 label =", label.item())
print("  requires_grad =", label.requires_grad, "-> 미분 대상이 아닌 상수입니다.")

[3] 정답값 label = 2.0
  requires_grad = False -> 미분 대상이 아닌 상수입니다.


In [4]:
cost = (pred - label)**2

print("[4] 손실 cost = (pred - label)^2")
print("  (-12 - 2)^2 = (-14)^2 =", cost.item())
print("  cost.grad_fn =", cost.grad_fn, "-> 이 스칼라가 역전파의 출발점입니다.")

[4] 손실 cost = (pred - label)^2
  (-12 - 2)^2 = (-14)^2 = 196.0
  cost.grad_fn = <PowBackward0 object at 0x74cb88592fe0> -> 이 스칼라가 역전파의 출발점입니다.


## `backward()` 로 그래디언트 계산

`cost.backward()` 를 호출하면 PyTorch 가 계산 그래프를 거꾸로 따라가며 `requires_grad=True` 인 모든 잎(leaf) 텐서에 대해 `∂cost/∂(텐서)` 를 자동 계산합니다. 결과는 각 텐서의 `.grad` 속성에 저장됩니다.

> 같은 그래프에 `backward()` 를 두 번 호출하면 기본적으로 그래프가 해제돼 에러가 납니다. 다시 호출하려면 `cost.backward(retain_graph=True)` 를 사용하세요.

### 그래디언트 읽기

`backward()` 가 끝나면 각 잎 텐서의 `.grad` 에 `∂cost/∂(텐서)` 가 채워져 있습니다. 아래 두 셀에서 `a.grad`, `b.grad` 를 확인합니다. (실제 학습이라면 `optimizer.step()` 이 이 값을 이용해 파라미터를 갱신합니다.)

In [5]:
cost.backward()

print("[5] cost.backward() 호출 완료")
print("  -> 계산 그래프를 거꾸로 따라가며 a, b 의 .grad 를 자동으로 채웠습니다.")

[5] cost.backward() 호출 완료
  -> 계산 그래프를 거꾸로 따라가며 a, b 의 .grad 를 자동으로 채웠습니다.


In [6]:
print("[6] a.grad = ∂cost/∂a =", a.grad)

[6] a.grad = ∂cost/∂a = tensor([-1008.])


In [7]:
print("[7] b.grad = ∂cost/∂b =", b.grad)

[7] b.grad = ∂cost/∂b = tensor([336.])


## 검산 — 손으로 미분해 보기

`pred = 3a³ − b²`, `cost = (pred − label)²` 이므로 연쇄법칙(chain rule)으로 각 변수의 그래디언트는 다음과 같습니다.

- `∂cost/∂a = 2(pred − label) · ∂pred/∂a = 2(pred − label) · 9a²`
- `∂cost/∂b = 2(pred − label) · ∂pred/∂b = 2(pred − label) · (−2b)`

`a=2, b=6, label=2` 를 대입하면 `pred = 24 − 36 = −12`, 따라서 `pred − label = −14` 이므로

- `∂cost/∂a = 2·(−14)·(9·2²) = 2·(−14)·36 = −1008`
- `∂cost/∂b = 2·(−14)·(−2·6) = 2·(−14)·(−12) = 336`

위 `backward()` 가 구한 **`a.grad = −1008`, `b.grad = 336` 과 정확히 일치**합니다. autograd 가 손으로 한 연쇄법칙과 똑같은 답을 자동으로 계산해 준다는 것을 확인할 수 있습니다.

아래 셀은 `∂pred/∂a = 9a² = 36` 부분만 따로 출력해 검산의 한 조각을 눈으로 확인하는 용도입니다.

In [8]:
print("[8] 부분 검산 ∂pred/∂a = 9a^2 =", (9*a**2).item())

[8] 부분 검산 ∂pred/∂a = 9a^2 = 36.0
